In [1]:
import sys
sys.path.append('../src/FluoreModel/')

import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from functools import partial
from transformers import AutoTokenizer
from torch import nn
import torch

import model
from gfp_datasets import SimpleDataset, collate_fn
import train_utils



BATCH_SIZE = 16
ESM_TYPE = "facebook/esm2_t33_650M_UR50D"


mutations_df = pd.read_csv("../data/data.csv")
mutations_df.Brightness = mutations_df.Brightness.astype('float32')


train_df = pd.read_csv("../data/train.csv")
X_train = train_df['seq']
y_train = train_df['Brightness']

val_df = pd.read_csv("../data/val.csv")
X_val = val_df['seq']
y_val = val_df['Brightness']


model = model.GFPRegressionModel([1280, 1200, 1100, 1000, 900, 800, 700, 600, 500, 400, 300, 200, 100, 50, 1], device='cuda', freeze_esm=False)
if torch.cuda.device_count() > 1:
    print(f"Используем {torch.cuda.device_count()} GPU!")
    model = nn.DataParallel(model) 

tokenizer = AutoTokenizer.from_pretrained(ESM_TYPE)
my_collate_fn = partial(collate_fn, tokenizer=tokenizer, max_length=1024)    

train_dataset = SimpleDataset(X_train.to_numpy(), y_train.to_numpy())
val_dataset = SimpleDataset(X_val.to_numpy(), y_val.to_numpy())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=my_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=my_collate_fn)

/trinity/home/d_ryabov/.conda/envs/newNucDPosIT/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Режим: ESM-2 дообучается
Используем 2 GPU!


In [ ]:
train_utils.train_model(model, train_loader, val_loader, save_path=None) 

/trinity/home/d_ryabov/.conda/envs/newNucDPosIT/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



📊 TensorBoard logs: ../logs/tensorboard/20260609_175607
🚀 Запустите: tensorboard --logdir=../logs/tensorboard


НАЧАЛО ОБУЧЕНИЯ
Устройство: cuda
Эпох: 50
Learning rate: 0.001
Weight decay: 0.0001
Patience: 15



Training:   0%|          | 0/3097 [00:00<?, ?it/s]/trinity/home/d_ryabov/.conda/envs/newNucDPosIT/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([16])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Training:   1%|          | 22/3097 [00:30<1:10:00,  1.37s/it, loss=20.4]   

In [ ]:
# def collate_fn(batch, tokenizer, max_length=1024):
#     """
#     Токенизация всего батча целиком (на CPU, как и должно быть)
#     """
#     sequences = list(batch[0])
#     targets = batch[1]
    
#     inputs = tokenizer(
#         sequences,
#         return_tensors="pt",
#         truncation=True,
#         padding=True,
#         max_length=max_length
#     )
    
#     return inputs, targets 

In [ ]:
for i in val_loader:
    break

In [ ]:
res = model.module(i[0])

In [ ]:
i[1]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.histplot(res.cpu().detach().numpy(), kde=True)
sns.histplot(i[1], kde=True)